In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
# from transformers import T5Tokenizer, T5ForConditionalGeneration
import transformers # requires transformers==4.35.2
device = torch.device('cuda:0')

In [2]:
print(transformers.__version__)

4.43.3


In [3]:
# draft_model_name = "deepseek-ai/deepseek-coder-1.3b-instruct"
# draft_model_name ="codellama/CodeLlama-7b-hf"
draft_model_name = "EleutherAI/pythia-1b"
draft_model = AutoModelForCausalLM.from_pretrained(draft_model_name, trust_remote_code=True, device_map="auto", torch_dtype=torch.float16)#, use_flash_attention_2=True)#, load_in_4bit=True)
print(draft_model.device)

cuda:0


In [4]:
# model_name = "deepseek-ai/deepseek-coder-6.7b-instruct"
# model_name="codellama/CodeLlama-70b-hf"
model_name = "EleutherAI/pythia-12b"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, device_map="auto", torch_dtype=torch.float16)
                                                   #, use_flash_attention_2=True)#, load_in_4bit=True)#  , use_flash_attention=True)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/49.4k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/9.81G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/9.93G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.11G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
from datasets import load_dataset

ds = load_dataset("abisee/cnn_dailymail", "1.0.0", split="test").select(list(range(100)))

In [6]:
import difflib

@torch.no_grad()
def find_candidate_pred_tokens(input_ids, max_ngram_size=3, num_pred_tokens=10):
    input_length = input_ids.size(1)

    # Ensure max_ngram_size and num_pred_tokens are valid
    if max_ngram_size <= 0 or num_pred_tokens <= 0 or max_ngram_size > input_length:
        raise ValueError("Invalid max_ngram_size or num_pred_tokens")

    for ngram_size in range(max_ngram_size, 0, -1):
        # Extract the last n tokens as our search ngram
        ngram = input_ids[0, -ngram_size:].tolist()

        # Create sliding windows of size ngram_size
        windows = input_ids.unfold(dimension=1, size=ngram_size, step=1)

        # Convert ngram to a tensor for comparison
        ngram_tensor = torch.tensor(ngram, device=input_ids.device).unsqueeze(0)

        # Find where the windows match the ngram
        matches = (windows == ngram_tensor).all(dim=2)

        # Get the indices of matches
        match_indices = matches.nonzero(as_tuple=True)[1]

        # Iterate through match indices to find a valid continuation
        for idx in match_indices:
            start_idx = idx + ngram_size
            end_idx = start_idx + num_pred_tokens
            # Ensure we don't go beyond the length of input_ids and avoid self-match
            # if end_idx <= input_length and start_idx < input_length - ngram_size:
            #     return input_ids[0, start_idx:end_idx]
            if start_idx < input_length - ngram_size:
                return input_ids[0, start_idx:min(end_idx, input_length)]

    # If no match is found, return an empty tensor
    return torch.tensor([100], dtype=torch.long, device=input_ids.device)

@torch.no_grad()
def find_candidate_pred_tokens_diff(input_ids, code_ids, orig_input_len=0, ngram_size=3, num_pred_tokens=10):
    # print(input_ids, code_ids)
    
    # start_time = time.perf_counter()
    input_length = input_ids.size(1)
    code_length = len(code_ids)

    # Ensure max_ngram_size and num_pred_tokens are valid
    if ngram_size <= 0 or ngram_size > input_length:
        raise ValueError("Invalid max_ngram_size or num_pred_tokens")

    sm = difflib.SequenceMatcher(None, code_ids, input_ids[0, orig_input_len:].tolist())
    
    deleted = added = changed = same = last_deleted = 0
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'replace':
            changed += i2 - i1
        elif tag == 'delete':
            deleted += i2 - i1
            last_deleted = i2 - i1
        elif tag == 'insert':
            added += j2 - j1
        elif tag == 'equal':
            same += i2 - i1
    
    approx_tokens_original = changed + deleted + same - last_deleted

    lookback_start = max(input_length - ngram_size, orig_input_len)
    search_ngram = input_ids[0, lookback_start:].tolist()

    for ngram_start in range(max(0, approx_tokens_original - ngram_size), len(code_ids)):
        # if there is a match, return the entire rest of the tokens.
        if ngram_start + len(search_ngram) >= len(code_ids):
            break
        if search_ngram == code_ids[ngram_start:ngram_start + len(search_ngram)]:
            return torch.tensor(code_ids[ngram_start + len(search_ngram):max(ngram_start + len(search_ngram) + num_pred_tokens, len(code_ids))], dtype=torch.long, device=input_ids.device)

    # If no match is found, return what the answer would be otherwise
    # print("Diff searching took: ", time.perf_counter() - start_time)
    return find_candidate_pred_tokens(input_ids, ngram_size, num_pred_tokens)
    # return torch.tensor([], dtype=torch.long, device=input_ids.device)


In [7]:
from transformers.generation.candidate_generator import CandidateGenerator, _crop_past_key_values
from transformers.generation.stopping_criteria import StoppingCriteria
from transformers.generation.configuration_utils import GenerationConfig
from typing import Tuple, Optional
import time


class NumRunsStoppingCriteria(StoppingCriteria):
    def __init__(self, max_num_runs=4):
        self.max_num_runs = 4
        self.num_runs = 0

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> torch.BoolTensor:
        self.num_runs += 1
        return self.num_runs >= self.max_num_runs
        
class ScoreStoppingCriteria:
    def __init__(self, min_score):
        self.min_score = min_score

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> torch.BoolTensor:
        if not(scores):
            # print("No scores")
            return False
        else:
            ...
            # print("Got scores scores stopping: ", scores[0].shape, len(scores))
        scores_tensor = torch.stack(scores, dim=0)
        softmax_scores = F.softmax(scores_tensor, 2)
        # print(softmax_scores)
        return (softmax_scores.max(dim=2).values < self.min_score).any().item()

def _get_default_candidate_generator_generator(generator: CandidateGenerator):
    def _get_candidate_generator(self, **kwargs):
        return generator
    return _get_candidate_generator

class TwoLayerLookupCandidateGenerator(CandidateGenerator):
    def __init__(self, tokenizer, prompt_tokens, draft_model, input_ids, use_score_check=False, min_score=0, scores_count=0, num_runs=4, num_pred_tokens=10):
        self.tokenizer = tokenizer
        self.prompt_tokens = prompt_tokens
        self.draft_model = draft_model
        self.input_ids = input_ids
        self.draft_model.generation_config.pad_token_id = tokenizer.pad_token_id
        
        self.past_key_values = None
        self.num_runs = num_runs

        self.start_token_index = self.input_ids.shape[-1]
        self.min_score = min_score
        self.scores_count = scores_count

        self.use_score_check = use_score_check
        self.num_pred_tokens_pld = num_pred_tokens
    
    def get_candidates(self, input_ids: torch.LongTensor) -> Tuple[torch.LongTensor, Optional[torch.FloatTensor]]:
        if self.past_key_values:
            self.past_key_values = _crop_past_key_values(self.draft_model, self.past_key_values, input_ids.shape[-1] - 1)

        stopping_criteria = [NumRunsStoppingCriteria(self.num_runs)]
        if self.use_score_check:
            stopping_criteria = [NumRunsStoppingCriteria(self.num_runs), 
                                 ScoreStoppingCriteria(self.min_score)]

        # if self.past_key_values:
        #     print(self.past_key_values[0][0].shape)

        if self.past_key_values: 
            generation = self.draft_model.generate(
                inputs=input_ids,
                attention_mask=torch.ones(input_ids.shape[-1], device=input_ids.device).unsqueeze(0),
                prompt_lookup_num_tokens=1,
                max_new_tokens=self.num_pred_tokens_pld,
                stopping_criteria=stopping_criteria,
                past_key_values=self.past_key_values,
                use_cache=True,
                output_scores=True,
                return_dict_in_generate=True
            )
        else:
            generation = self.draft_model.generate(
                inputs=input_ids,
                attention_mask=torch.ones(input_ids.shape[-1], device=input_ids.device).unsqueeze(0),
                prompt_lookup_num_tokens=1,
                max_new_tokens=self.num_pred_tokens_pld,
                stopping_criteria=stopping_criteria,
                use_cache=True,
                output_scores=True,
                return_dict_in_generate=True
            )
        # print("Scores: ", generation.scores)

        self.pred_tokens_count = generation.sequences.shape[-1] - input_ids.shape[-1]
        self.past_key_values = generation.past_key_values
        self.past_top_scores = torch.stack(generation.scores, dim=1).max(dim=1).values[0]

        return generation.sequences, torch.stack(generation.scores, dim=1)

    def update_candidate_strategy(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, num_matches: int):
        if num_matches == self.pred_tokens_count:
            if self.scores_count == 0:
                self.min_score = 0
            else:
                self.min_score = (self.scores_count / self.scores_count + 1) * (self.min_score)
        else:
            if self.scores_count == 0:
                self.min_score = self.past_top_scores[-num_matches]
            else:
                self.min_score = (self.scores_count / (self.scores_count + 1)) * (self.min_score) + (1 / (self.scores_count + 1)) * (self.past_top_scores[-1])
        self.scores_count += 1
        pass 

In [8]:
def print_update(dictionary):
    for key in dictionary:
        print("\t", key, ": ", dictionary[key][-1])
    print("======")

In [9]:
from tqdm import tqdm
from transformers import TextStreamer
from rapidfuzz.distance import Levenshtein

lookup_tokens = [5, 10, 15]
# lookup_tokens = [40]
stats = {lt: {"method": [], "method_with_score_cutoff": [], "assisted": [], "pld": [], "regular": [], "generated_tokens": []} for lt in lookup_tokens}

global_min_score = 0
global_scores_count = 0

regular_get_candidate_generator = model._get_candidate_generator

for lt in lookup_tokens:
    for row in tqdm(ds):
        input_text = f"summarize: {row['article']}"
        # inputs = tokenizer.apply_chat_template([
        #         {
        #             "role": "user",
        #             "content": input_text
        #         },
        #     ], tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)    
        inputs = tokenizer.encode(input_text, return_tensors="pt").to(model.device)
        starting_input_tokens = inputs.shape[-1]
        
        max_new_tokens = 500

        model._get_candidate_generator = (regular_get_candidate_generator).__get__(model, type(model))

        # Use HuggingFace assisted decoding
        start_time = time.perf_counter()
        assisted_output = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            return_dict_in_generate=True,
            output_scores=True,
            assistant_model=draft_model
        )
        end_time = time.perf_counter()
        stats[lt]["assisted"].append(end_time - start_time)
    
        # Use HuggingFace prompt lookup decoding
        start_time = time.perf_counter()
        pld_output = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            return_dict_in_generate=True,
            output_scores=True,
            prompt_lookup_num_tokens=lt
        )
        end_time = time.perf_counter()
        stats[lt]["pld"].append(end_time - start_time)
    
        # Use regular HuggingFace text generation
        start_time = time.perf_counter()
        regular_outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            return_dict_in_generate=True,
            output_scores=True
        )
        end_time = time.perf_counter()
    
        stats[lt]["regular"].append(end_time - start_time)
        new_text = tokenizer.batch_decode(regular_outputs.sequences[:, starting_input_tokens:])[0]

        # print(row['before'], new_text)
        stats[lt]["generated_tokens"].append(regular_outputs.sequences.shape[-1])

        # Two Layer Lookup Candidate Generator with Score Check
        two_layer_candidate_generator = TwoLayerLookupCandidateGenerator(
            tokenizer,
            inputs.shape[-1],
            draft_model,
            inputs,
            use_score_check=True,
            min_score=global_min_score,
            scores_count=global_scores_count,
            num_pred_tokens=lt
        )
        model._get_candidate_generator = (_get_default_candidate_generator_generator(two_layer_candidate_generator)).__get__(model, type(model))
    
        global_min_score = two_layer_candidate_generator.min_score
        global_scores_count = two_layer_candidate_generator.scores_count
        start_time = time.perf_counter()
        test_out = model.generate(
            inputs=inputs,
            prompt_lookup_num_tokens=1,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            # streamer=TextStreamer(tokenizer)
        )
        end_time = time.perf_counter()
        stats[lt]["method_with_score_cutoff"].append(end_time - start_time)

        # Two Layer Lookup Candidate Generator without Score Check
        two_layer_candidate_generator = TwoLayerLookupCandidateGenerator(
            tokenizer,
            inputs.shape[-1],
            draft_model,
            inputs,
            use_score_check=False,
            min_score=global_min_score,
            scores_count=global_scores_count,
            num_pred_tokens=lt
        )
        model._get_candidate_generator = (_get_default_candidate_generator_generator(two_layer_candidate_generator)).__get__(model, type(model))
    
        global_min_score = two_layer_candidate_generator.min_score
        global_scores_count = two_layer_candidate_generator.scores_count
        start_time = time.perf_counter()
        test_out = model.generate(
            inputs=inputs,
            prompt_lookup_num_tokens=1,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            # streamer=TextStreamer(tokenizer)
        )
        end_time = time.perf_counter()
        stats[lt]["method"].append(end_time - start_time)

        print_update(stats[lt])

print(stats)

  0%|                                                                                                                                                    | 0/100 [00:00<?, ?it/s]


ValueError: Make sure the main and assistant model use the same tokenizer

In [ ]:
import json
stats_file = open("stats_qwen_dailymail.json", "w+")
stats_file.write(json.dumps(stats))
stats_file.close()